# Basin join exploration

Scratch notebook for exploring how the DataBasin CA Substations 2022 reference dataset
aligns with the utility substation sources (PGE, SCE, SDGE).

These CSVs were produced by `scripts/compare_substations.py`.

**Two join strategies were used:**

| Section | File pattern | Method | Coverage |
|---------|-------------|--------|----------|
| B | `cmp_B_*_name_join.csv` | Normalised-name exact match | Only substations whose names match |
| C | `cmp_C_*_spatial_join.csv` | Nearest-neighbour (KDTree on unit sphere) | Every utility substation with coords gets a basin neighbour |

**Name normalisation** strips P.T. suffix, the word 'substation', punctuation, and extra whitespace, then lowercases.
This is the same `norm()` function used in `process_substations_clean.py`.

In [ ]:
from pathlib import Path
import pandas as pd

CHECKS = Path('../data/checks')

---
## Section B — Name-join files

Each row is a **substation that matched by normalised name** between the utility source and basin.

Columns:
- `name_norm` — the normalised name used for the join key
- `name_raw_basin` / `name_raw_util` — original strings from each source before normalisation
- `lat_basin`, `lon_basin` — DataBasin 2022 coordinates
- `lat_util`, `lon_util` — utility source coordinates
- `dist_km` — haversine distance between the two coordinate pairs (how far apart the coords are for the *same* substation name)

**Note:** these files only contain substations that *did* match by name.
Substations that are in the utility source but not in basin (by name) are in the corresponding `*_only_in_source.csv`.

### PGE attrs — name join
706 unique PGE substations in attrs; 576 matched basin by name.

In [ ]:
b_pge_attrs = pd.read_csv(CHECKS / 'cmp_B_pge_attrs_name_join.csv')
print(b_pge_attrs.shape)
b_pge_attrs.head()

### PGE loads — name join
664 unique PGE substations in loads; 550 matched basin by name.

In [ ]:
b_pge_loads = pd.read_csv(CHECKS / 'cmp_B_pge_loads_name_join.csv')
print(b_pge_loads.shape)
b_pge_loads.head()

### SCE attrs_alt (ICA Layer) — name join
734 unique SCE substations in the alt attribute file; 525 matched basin by name.
This is the preferred SCE coordinate source (reliable lat/lon for ~735 substations).

In [ ]:
b_sce_alt = pd.read_csv(CHECKS / 'cmp_B_sce_attrs_alt_name_join.csv')
print(b_sce_alt.shape)
b_sce_alt.head()

### SCE loads scrape — name join
745 unique substations in the scrape rows of `sce_combined_raw.csv` (scrape has lat/lon; bulk does not).
512 matched basin by name.

In [ ]:
b_sce_scrape = pd.read_csv(CHECKS / 'cmp_B_sce_loads_scrape_name_join.csv')
print(b_sce_scrape.shape)
b_sce_scrape.head()

### SDGE attrs — name join
107 unique SDGE substations in attrs (after excluding failures); 93 matched basin by name.

In [ ]:
b_sdge_attrs = pd.read_csv(CHECKS / 'cmp_B_sdge_attrs_name_join.csv')
print(b_sdge_attrs.shape)
b_sdge_attrs.head()

### SDGE loads — name join
99 unique SDGE substations in loads; 87 matched basin by name.

In [ ]:
b_sdge_loads = pd.read_csv(CHECKS / 'cmp_B_sdge_loads_name_join.csv')
print(b_sdge_loads.shape)
b_sdge_loads.head()

---
## Section C — Spatial join files

Each row is a utility substation matched to its **nearest basin point** by great-circle distance,
regardless of whether the names agree.

Columns:
- `util_name` / `util_norm` — utility substation name (raw and normalised)
- `util_lat`, `util_lon` — utility source coordinates
- `basin_name` / `basin_norm` — the nearest basin substation's name
- `basin_lat`, `basin_lon` — basin coordinates
- `dist_km` — haversine distance to the nearest basin point
- `name_match` — True if `util_norm == basin_norm` (names agree after normalisation)

**Every** utility substation with coords appears here (no filtering by name).
A small `dist_km` with `name_match=False` is the interesting case: the coordinates agree
but the names don't — these are candidates for a fuzzy or alias join.

### PGE attrs — spatial join

In [ ]:
c_pge_attrs = pd.read_csv(CHECKS / 'cmp_C_pge_attrs_spatial_join.csv')
print(c_pge_attrs.shape)
c_pge_attrs.head()

### PGE loads — spatial join

In [ ]:
c_pge_loads = pd.read_csv(CHECKS / 'cmp_C_pge_loads_spatial_join.csv')
print(c_pge_loads.shape)
c_pge_loads.head()

### SCE attrs_alt — spatial join

In [ ]:
c_sce_alt = pd.read_csv(CHECKS / 'cmp_C_sce_attrs_alt_spatial_join.csv')
print(c_sce_alt.shape)
c_sce_alt.head()

### SCE loads scrape — spatial join

In [ ]:
c_sce_scrape = pd.read_csv(CHECKS / 'cmp_C_sce_loads_scrape_spatial_join.csv')
print(c_sce_scrape.shape)
c_sce_scrape.head()

### SDGE attrs — spatial join

In [ ]:
c_sdge_attrs = pd.read_csv(CHECKS / 'cmp_C_sdge_attrs_spatial_join.csv')
print(c_sdge_attrs.shape)
c_sdge_attrs.head()

### SDGE loads — spatial join

In [ ]:
c_sdge_loads = pd.read_csv(CHECKS / 'cmp_C_sdge_loads_spatial_join.csv')
print(c_sdge_loads.shape)
c_sdge_loads.head()